In [ ]:
from pathlib import Path
from ultralytics.models.yolo import YOLO
import cv2

data_dir = Path("../untracked")
video_path = data_dir / "vz_20s.mp4"

In [ ]:
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS) or 30
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

output_yolo_path = data_dir / "pose_output.mp4"
model = YOLO("yolo11m-pose.pt")
fourcc = cv2.VideoWriter_fourcc(*"avc1")
# writer = cv2.VideoWriter(str(output_yolo_path), fourcc, fps, (w, h), isColor=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

head_centers = []  # Will store [(x, y), ...] for each frame
shoulder_distances = []  # Will store distances for each frame
frame_indices: list[int] = []

frame_idx = -1
for result in model.track(source=str(video_path), stream=True, tracker="bytetrack.yaml"):
    # annotated = result.plot()  # draw poses + boxes
    # writer.write(annotated)
    frame_idx += 1

    # Get the original frame
    # frame = result.orig_img.copy()

    if result.keypoints is not None and result.keypoints.data is not None:
        keypoints: np.ndarray = np.array(result.keypoints.data)[0]
        # print(keypoints)

        # left_ear = keypoints[3][:2] if keypoints[3][2] > 0 else None
        # right_ear = keypoints[4][:2] if keypoints[4][2] > 0 else None
        left_eye = keypoints[1][:2] if keypoints[1][2] > 0 else None
        right_eye = keypoints[2][:2] if keypoints[2][2] > 0 else None
        nose = keypoints[0][:2] if keypoints[0][2] > 0 else None

        head_points = []
        if left_eye is not None:
            head_points.append(left_eye)
        if right_eye is not None:
            head_points.append(right_eye)
        if nose is not None:
            head_points.append(nose)
        
        if head_points:
            head_center = np.mean(head_points, axis=0).astype(int)
            head_centers.append(tuple(head_center.astype(int)))
            frame_indices.append(frame_idx)
        
        left_shoulder = keypoints[5][:2] if keypoints[5][2] > 0 else None
        right_shoulder = keypoints[6][:2] if keypoints[6][2] > 0 else None

        if left_shoulder is not None and right_shoulder is not None:
            distance = np.linalg.norm(left_shoulder - right_shoulder)
            shoulder_distances.append(distance)
        

        # if left_eye is not None:
        #     left_eye_coords = tuple(left_eye.astype(int))
        #     cv2.circle(frame, left_eye_coords, 5, (255, 0, 0), -1)  # Blue circle for left ear
        #     cv2.putText(frame, "L", (left_eye_coords[0]+5, left_eye_coords[1]-5), 
        #                 cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)
            
        # if right_eye is not None:
        #     right_eye_coords = tuple(right_eye.astype(int))
        #     cv2.circle(frame, right_eye_coords, 5, (0, 255, 0), -1)  # Green circle for right ear
        #     cv2.putText(frame, "R", (right_eye_coords[0]+5, right_eye_coords[1]-5), 
        #                 cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        # frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # clear_output(wait=True)
        # plt.figure(figsize=(12, 8))
        # plt.imshow(frame_rgb)

        # break


# writer.release()

In [ ]:
shoulder_distance_ave = np.mean(shoulder_distances)
shoulder_distance_ave

In [ ]:
from typing import Sequence
from utils import nw_smoothing

def smooth(frms: Sequence[int], xy:Sequence[tuple[np.float32, np.float32]], sigma:float=1) -> list[tuple[np.float32, np.float32]]:
     xs = nw_smoothing(frms, [xy_i[0] for xy_i in xy])
     ys = nw_smoothing(frms, [xy_i[1] for xy_i in xy])
     return list(zip(xs, ys))

smoothed_centers: list[tuple[np.float32, np.float32]] = smooth(frame_indices, head_centers)

In [ ]:
output_cropped_path = data_dir / "head_cropped.mp4"
fourcc = cv2.VideoWriter_fourcc(*"avc1")
crop_size = int(shoulder_distance_ave * 0.7)
writer = cv2.VideoWriter(str(output_cropped_path), fourcc, fps, (crop_size, crop_size), isColor=True)


# Initialize previous position for missing frames
prev_smoothed_center = None

cap = cv2.VideoCapture(str(video_path))

# Process frames in order
for frame_idx in sorted(frame_indices):
    print(f"frame IDX {frame_idx} out of {len(frame_indices)}")
    # Set video to specific frame
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    
    if not ret:
        continue
    
    # Get current smoothed center
    if frame_idx < len(smoothed_centers):
        current_smoothed = smoothed_centers[frame_idx]
        
        # Skip if first frames are missing (both current and previous are None)
        if current_smoothed is None and prev_smoothed_center is None:
            continue
            
        # Use previous non-missing position if current is missing
        if current_smoothed is None and prev_smoothed_center is not None:
            current_smoothed = prev_smoothed_center
        elif current_smoothed is not None:
            prev_smoothed_center = current_smoothed
    else:
        # Frame index out of range, use previous if available
        if prev_smoothed_center is None:
            continue
        current_smoothed = prev_smoothed_center
    
    # Calculate crop region
    center_x, center_y = current_smoothed
    x_start = int(max(0, center_x - crop_size // 2))
    x_end = int(min(w, x_start + crop_size))
    y_start = int(max(0, center_y - crop_size // 2))
    y_end = int(min(h, y_start + crop_size))
    
    # Adjust if crop window goes out of bounds
    if x_end - x_start < crop_size:
        if x_start == 0:
            x_end = min(w, crop_size)
        else:
            x_start = max(0, w - crop_size)
    if y_end - y_start < crop_size:
        if y_start == 0:
            y_end = min(h, crop_size)
        else:
            y_start = max(0, h - crop_size)
    
    # Crop the frame
    cropped_frame = frame[y_start:y_end, x_start:x_start + crop_size]
    
    # Plot centers on the cropped frame
    if frame_idx < len(head_centers) and head_centers[frame_idx] is not None:
        # Adjust head center coordinates relative to crop window
        head_x, head_y = head_centers[frame_idx]
        head_x_rel = int(head_x - x_start)
        head_y_rel = int(head_y - y_start)
        if 0 <= head_x_rel < w and 0 <= head_y_rel < h:
            cv2.circle(cropped_frame, (head_x_rel, head_y_rel), 5, (0, 255, 0), -1)  # Green for non-smoothed
    
    # Plot smoothed center
    smooth_x_rel = int(center_x - x_start)
    smooth_y_rel = int(center_y - y_start)
    if 0 <= smooth_x_rel < w and 0 <= smooth_y_rel < h:
        cv2.circle(cropped_frame, (smooth_x_rel, smooth_y_rel), 5, (0, 0, 255), -1)  # Red for smoothed
        # Add a small cross for better visibility
        cv2.drawMarker(cropped_frame, (smooth_x_rel, smooth_y_rel), (0, 0, 255), cv2.MARKER_CROSS, 10, 1)
    
    # Write the frame
    writer.write(cropped_frame)

    # if frame_idx > 10: break

# Clean up
cap.release()
writer.release()
cv2.destroyAllWindows()